# Beyond-Memorization Style Per-Character Attention Maps

**Goal:** Reproduce the per-character attention-map visualization from
[*Beyond Memorization: Training-Free Style Mixing for Variability in Handwritten Text Generation*](https://github.com/aniketntnu/Beyond-Memorization)
(Gurav, Chanda, Krishnan — ICDAR 2025) for the sample images in `notebook/sample_images/`.

---

## How the original repo does it (`utils/saveAttentionMaps.py`)

Beyond-Memorization is a **generation** model (WordStylist diffusion U-Net). Its U-Net contains
**cross-attention** layers where each image spatial position (query) attends to each
character embedding (key/value). The cross-attention tensor has shape
`[B, heads, H*W, MAX_CHARS]`, so slicing `attn[:, :, :, c]` directly gives the spatial
heatmap for character `c`. The repo then:
1. normalizes the per-character map to `[0, 1]`,
2. thresholds at `mean + sigma*std`,
3. runs connected-component labelling (`scipy.ndimage.label`),
4. keeps the **strongest blob**, draws its bounding box, and
5. overlays an `inferno` heatmap (alpha 0.5) on the word image.

## How we adapt it to our HTR model (recognition, not generation)

Our `vit_rgts` model is a **recognition** model that uses **self-attention** (token ↔ token),
not image→character cross-attention. There is no `MAX_CHARS` axis to slice. We bridge the gap
with a **CTC-aligned** extraction (same idea, same blob/threshold/overlay maths):

| Beyond-Memorization (generation) | Our HTR pipeline (recognition) |
|----------------------------------|--------------------------------|
| Cross-attention `image → char` | Self-attention `token ↔ token` |
| `attn[:, :, :, c]` is char map | CTC greedy decode maps each char → timestep span; we read self-attention rows at those timesteps |
| Threshold + blob + bbox + inferno overlay | **identical** post-processing |

Pipeline: `forward_explain` → CTC decode → per-char timestep span → self-attention over patches →
upsample to image width → threshold + strongest blob → `inferno` overlay + bbox.

## Step 0 — Imports & configuration

In [ ]:
import os, sys, json
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import label as ndlabel
from omegaconf import OmegaConf

# Make the repo root importable (this notebook lives in notebook/)
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebook':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from models import HTRNet
from utils.preprocessing import load_image, preprocess

# ---- Paths / config ----
RUN_DIR     = REPO_ROOT / 'saved_models' / 'experiments' / 'run_104'   # vit_rgts (CNN stem + 2 registers)
MODEL_PATH  = RUN_DIR / 'model.pt'
CONFIG_PATH = RUN_DIR / 'config.json'
CLASSES_NPY = REPO_ROOT / 'data' / 'IAM' / 'processed_lines' / 'classes.npy'
SAMPLE_DIR  = REPO_ROOT / 'notebook' / 'sample_images'
SAVE_DIR    = REPO_ROOT / 'outputs' / 'attention_maps' / 'notebook_beyond_memorization'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Blob threshold strength (Beyond-Memorization uses mean + sigma*std)
SIGMA     = 1.0
LAYER_IDX = -1   # which transformer layer's self-attention to read (-1 = last)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device      :', DEVICE)
print('Model path  :', MODEL_PATH)
print('Exists      :', MODEL_PATH.exists())

## Step 1 — Load the character set and the trained HTR model

`classes.npy` holds the 79 characters; CTC adds a blank at index 0, so `num_classes = 80`.
We rebuild the architecture from the run's `config.json` (`vit_rgts`, CNN stem, `num_registers=2`)
and load the trained weights.

In [ ]:
cfg = OmegaConf.load(str(CONFIG_PATH))

classes = list(np.load(CLASSES_NPY, allow_pickle=True))
i2l = {str(i): str(c) for i, c in enumerate(classes)}   # CTC index (offset by blank) -> letter
num_classes = len(classes) + 1                           # +1 for CTC blank at index 0
NUM_REGISTERS = int(cfg.arch.num_registers)

model = HTRNet(cfg.arch, num_classes).to(DEVICE)
state = torch.load(str(MODEL_PATH), map_location=DEVICE)
model.load_state_dict(state)
model.eval()

print(f'Loaded {model.arch_type} | classes={num_classes} (79 + blank) | registers={NUM_REGISTERS}')

## Step 2 — Load and preprocess one sample image

`load_image` reads + inverts the grayscale word (ink → high value), and `preprocess`
aspect-ratio resizes and pads it to the model input size `128×1024`. We display the
padded canvas with `gray_r` so ink appears dark on white — this is the canvas all
heatmaps will be overlaid on.

In [ ]:
IMG_HEIGHT = int(cfg.preproc.image_height)   # 128
IMG_WIDTH  = int(cfg.preproc.image_width)    # 1024

def load_sample(name):
    """Return (preprocessed_canvas[H,W], input_tensor[1,1,H,W])."""
    path = SAMPLE_DIR / name
    raw = load_image(str(path))
    proc = preprocess(raw, (IMG_HEIGHT, IMG_WIDTH))
    tensor = torch.from_numpy(proc).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
    return proc, tensor

SAMPLE = 'a01-038-12.png'   # ground-truth word: 'talks.'
image_np, image_tensor = load_sample(SAMPLE)

plt.figure(figsize=(12, 1.8))
plt.imshow(image_np, cmap='gray_r', aspect='auto')
plt.title(f'Preprocessed input: {SAMPLE}')
plt.axis('off')
plt.show()
print('Input tensor:', tuple(image_tensor.shape))

## Step 3 — Forward pass with attention extraction (`forward_explain`)

Returns the CTC `logits`, the register tokens, the per-layer self-attention maps
`[B, H, S, S]` (with `S = R + P`), token norms, and the patch grid `(Hp, Wp)`.
For our CNN-stem `vit_rgts` the grid is `1×128`, i.e. 128 horizontal patch tokens.

In [ ]:
with torch.no_grad():
    logits, reg_tokens, attn_maps, token_norms, grid = model.forward_explain(image_tensor)

Hp, Wp = grid
S = attn_maps[LAYER_IDX].shape[-1]
print(f'logits        : {tuple(logits.shape)}   (T, B, num_classes)')
print(f'num layers    : {len(attn_maps)}')
print(f'attn[{LAYER_IDX}] shape : {tuple(attn_maps[LAYER_IDX].shape)}   (B, H, S, S)')
print(f'tokens S      : {S} = {NUM_REGISTERS} registers + {S - NUM_REGISTERS} patches')
print(f'patch grid    : Hp={Hp}, Wp={Wp}')

## Step 4 — CTC greedy decode → per-character timestep spans

This is the bridge that replaces the generation model's `MAX_CHARS` axis. Greedy CTC
decoding collapses repeats and removes blanks; while doing so we record, for every output
character, the **span of encoder timesteps** `(start, end)` it occupied. Those timesteps are
the patch positions that "belong" to that character — exactly what we read attention from.

In [ ]:
def ctc_greedy_decode(logits_tb_c, i2l, blank_idx=0):
    """Greedy CTC decode. Returns (text, char_spans) where each span is (char, start_t, end_t)."""
    probs = logits_tb_c.softmax(dim=-1)
    pred = probs.argmax(dim=-1).cpu().numpy()   # [T]

    spans = []
    prev = -1
    cur_char, cur_start = None, None
    for t, idx in enumerate(pred):
        if idx == blank_idx:
            if cur_char is not None:
                spans.append((cur_char, cur_start, t - 1)); cur_char = None
            prev = idx; continue
        if idx == prev:
            prev = idx; continue
        if cur_char is not None:
            spans.append((cur_char, cur_start, t - 1))
        cur_char = i2l.get(str(idx - 1), '?')   # -1: undo CTC blank offset
        cur_start = t; prev = idx
    if cur_char is not None:
        spans.append((cur_char, cur_start, len(pred) - 1))

    text = ''.join(c for c, _, _ in spans)
    return text, spans

text, char_spans = ctc_greedy_decode(logits[:, 0, :], i2l)
print(f"Decoded text : '{text}'  ({len(char_spans)} characters)")
for c, s, e in char_spans:
    print(f"   '{c}'  timesteps {s:>3}–{e:<3}")

## Step 5 — Per-character attention vector

For a character occupying timesteps `[start, end]`, we take the self-attention rows at those
patch positions (registers excluded), average over heads and over the character's timesteps,
and get a 1-D vector of length `P` (128) describing *where along the width that character
looks*. We then upsample it to the full image width so it can be overlaid.

In [ ]:
def char_attention_1d(attn_maps, char_span, num_registers, layer_idx=-1):
    """1-D patch attention [P] for one character (self-attention, CTC-aligned)."""
    attn = attn_maps[layer_idx][0]                 # [H, S, S]
    attn_avg = attn.mean(dim=0).cpu().numpy()      # [S, S] mean over heads
    patch_attn = attn_avg[num_registers:, num_registers:]   # [P, P] drop registers

    _, start, end = char_span
    P = patch_attn.shape[0]
    rows = [p for p in range(start, end + 1) if p < P]
    if not rows:
        return None
    return patch_attn[rows, :].mean(axis=0)        # [P]

def upsample_1d_to_width(attn_1d, width):
    """Repeat a 1-D vector up to image width."""
    rep = int(np.ceil(width / len(attn_1d)))
    hm = np.repeat(attn_1d, max(1, rep))
    if len(hm) < width:
        hm = np.pad(hm, (0, width - len(hm)), mode='edge')
    return hm[:width]

# Demo for the first character
demo = char_attention_1d(attn_maps, char_spans[0], NUM_REGISTERS, LAYER_IDX)
print(f"Char '{char_spans[0][0]}' -> 1D attention length {len(demo)}, peak at patch {int(demo.argmax())}")

## Step 6 — Strongest-blob detection (identical to Beyond-Memorization)

Normalize to `[0,1]`, threshold at `mean + sigma*std`, label connected components
(`scipy.ndimage.label`), keep the blob with the highest total attention, and return its
binary mask + bounding box. This is the exact `find_max_activation_above_threshold` /
`get_blob_centroids` logic from `saveAttentionMaps.py`.

In [ ]:
def find_strongest_blob(heatmap_2d, sigma=1.0):
    """Threshold (mean+sigma*std) -> connected components -> strongest blob + bbox."""
    normed = (heatmap_2d - heatmap_2d.min()) / (heatmap_2d.max() - heatmap_2d.min() + 1e-8)
    binary = normed >= (normed.mean() + sigma * normed.std())
    labeled, n = ndlabel(binary)
    if n == 0:
        return None
    best_lbl, best_sum = None, -1
    for lbl in range(1, n + 1):
        s = normed[labeled == lbl].sum()
        if s > best_sum:
            best_sum, best_lbl = s, lbl
    mask = labeled == best_lbl
    rows, cols = np.where(mask)
    return {'mask': mask,
            'bbox': (rows.min(), cols.min(), rows.max(), cols.max()),
            'centroid': (rows.mean(), cols.mean())}

## Step 7 — Per-character heatmap overlay (the Beyond-Memorization figure)

For each decoded character we build the full-resolution heatmap, run blob detection, and
render the word image with a semi-transparent `inferno` overlay plus the strongest-blob
bounding box — reproducing the per-character figures the repo saves under `attentionMaps/`.

In [ ]:
def per_character_maps(image_np, attn_maps, char_spans, num_registers,
                       layer_idx=-1, sigma=1.0):
    """Build (char, heatmap_2d, blob) for every decoded character."""
    H, W = image_np.shape[:2]
    results = []
    for ci, span in enumerate(char_spans):
        a1d = char_attention_1d(attn_maps, span, num_registers, layer_idx)
        if a1d is None:
            continue
        hm = upsample_1d_to_width(a1d, W)
        hm2d = np.tile(hm, (H, 1))
        hm2d = (hm2d - hm2d.min()) / (hm2d.max() - hm2d.min() + 1e-8)
        blob = find_strongest_blob(hm2d, sigma=sigma)
        results.append({'char': span[0], 'idx': ci, 'span': span[1:],
                        'heatmap': hm2d, 'blob': blob})
    return results

def show_char_map(image_np, res, sigma=SIGMA):
    fig, ax = plt.subplots(figsize=(12, 1.8))
    ax.imshow(image_np, cmap='gray_r', aspect='auto')
    ax.imshow(res['heatmap'], cmap='inferno', alpha=0.5, aspect='auto')
    if res['blob'] is not None:
        r0, c0, r1, c1 = res['blob']['bbox']
        ax.add_patch(mpatches.Rectangle((c0, r0), c1 - c0, r1 - r0,
                     linewidth=2, edgecolor='lime', facecolor='none'))
    ax.set_title(f"Attention Map for Character '{res['char']}'  (timesteps {res['span'][0]}–{res['span'][1]})",
                 fontsize=11, fontweight='bold')
    ax.axis('off')
    plt.show()

char_results = per_character_maps(image_np, attn_maps, char_spans, NUM_REGISTERS, LAYER_IDX, SIGMA)
print(f"Word '{text}' -> {len(char_results)} character maps\n")
for res in char_results:
    show_char_map(image_np, res)

## Step 8 — Fig. 5-style grid (word-level + per-character)

Column 0 shows the **word-level** attention `A` (sum of all character maps); the remaining
columns show each per-character map `A_c` with its blob box — the layout used in the paper's
Figure 5.

In [ ]:
def fig5_row(image_np, text, char_results, sigma=SIGMA, save_path=None):
    ncols = len(char_results) + 1
    fig, axes = plt.subplots(1, ncols, figsize=(2.2 * ncols, 2.0), squeeze=False)
    axes = axes[0]

    # Col 0: word-level attention = sum of character heatmaps
    word_hm = np.sum([r['heatmap'] for r in char_results], axis=0)
    word_hm = (word_hm - word_hm.min()) / (word_hm.max() - word_hm.min() + 1e-8)
    axes[0].imshow(image_np, cmap='gray_r', aspect='auto')
    axes[0].imshow(word_hm, cmap='inferno', alpha=0.5, aspect='auto')
    axes[0].set_title(f"A: '{text}'", fontsize=10, fontweight='bold')
    axes[0].axis('off')

    for j, res in enumerate(char_results):
        ax = axes[j + 1]
        ax.imshow(image_np, cmap='gray_r', aspect='auto')
        ax.imshow(res['heatmap'], cmap='inferno', alpha=0.5, aspect='auto')
        if res['blob'] is not None:
            r0, c0, r1, c1 = res['blob']['bbox']
            ax.add_patch(mpatches.Rectangle((c0, r0), c1 - c0, r1 - r0,
                         linewidth=1.5, edgecolor='lime', facecolor='none'))
        ax.set_title(f"'{res['char']}'", fontsize=10)
        ax.axis('off')

    plt.tight_layout(pad=0.3)
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()

fig5_row(image_np, text, char_results,
         save_path=str(SAVE_DIR / f'fig5_{SAMPLE.replace(".png", "")}.png'))

## Step 9 — Run the full pipeline over several sample images

Wrap everything into one helper and process a list of word images. Per-character PNGs and
the Fig. 5 grid are written under `outputs/attention_maps/notebook_beyond_memorization/`.

In [ ]:
def run_image(name, layer_idx=LAYER_IDX, sigma=SIGMA, save=True, show_grid=True):
    img_np, tensor = load_sample(name)
    with torch.no_grad():
        lg, _, amaps, _, _ = model.forward_explain(tensor)
    txt, spans = ctc_greedy_decode(lg[:, 0, :], i2l)
    if not txt:
        print(f'{name}: empty decode, skipped'); return None
    cres = per_character_maps(img_np, amaps, spans, NUM_REGISTERS, layer_idx, sigma)

    out_dir = SAVE_DIR / name.replace('.png', '')
    if save:
        out_dir.mkdir(parents=True, exist_ok=True)
        for res in cres:
            fig, ax = plt.subplots(figsize=(12, 1.8))
            ax.imshow(img_np, cmap='gray_r', aspect='auto')
            ax.imshow(res['heatmap'], cmap='inferno', alpha=0.5, aspect='auto')
            if res['blob'] is not None:
                r0, c0, r1, c1 = res['blob']['bbox']
                ax.add_patch(mpatches.Rectangle((c0, r0), c1 - c0, r1 - r0,
                             linewidth=2, edgecolor='lime', facecolor='none'))
            ax.set_title(f"'{res['char']}'", fontsize=11, fontweight='bold')
            ax.axis('off')
            fig.savefig(out_dir / f"char_{res['idx']:02d}_{res['char']}.png",
                        dpi=150, bbox_inches='tight', pad_inches=0.05)
            plt.close(fig)

    print(f"{name}: '{txt}' ({len(cres)} chars)")
    if show_grid:
        fig5_row(img_np, txt, cres,
                 save_path=str(out_dir / 'fig5_grid.png') if save else None)
    return txt

# Short single-word samples work best for clean per-character localization
WORD_SAMPLES = ['a01-038-12.png', 'a01-091-10.png', 'a01-096u-10.png',
                'a06-110-08.png', 'r06-137-10.png']
for nm in WORD_SAMPLES:
    run_image(nm)

print('\nSaved to:', SAVE_DIR)

## Notes & knobs

- **`SIGMA`** (Step 0): higher → stricter blob threshold (smaller, sharper regions).
- **`LAYER_IDX`**: which transformer layer's self-attention to read. The last layer (`-1`)
  is most character-discriminative; mid layers (e.g. `2`/`3`) are more diffuse.
- **Grid is `1×128`**: the CNN stem produces horizontal patch tokens, so attention varies
  along the width only — the heatmap is tiled vertically over the word's height.
- **Recognition vs generation**: localization quality depends on the CTC alignment. Cross-attention
  in the original generation model gives a direct char axis; our self-attention + CTC span is the
  faithful recognition-side equivalent.
- A ready-made script version of this exists at
  `scripts/postprocessing/beyond_memorization_viz.py` (adds multi-model register comparison).